In [ ]:
# Terminal
# pip install scikit-learn==1.4.2
# Restart Kernel

In [1]:
pip install xgboost

  Using cached xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl.metadata (2.1 kB)
  Using cached nvidia_nccl_cu12-2.29.7-py3-none-manylinux_2_18_x86_64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl (131.7 MB)
Using cached nvidia_nccl_cu12-2.29.7-py3-none-manylinux_2_18_x86_64.whl (293.6 MB)

[notice] A new release of pip is available: 24.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, KFold

def tune_log(X_train, y_train, missing_rate, imputer, random_state=57):
    """
    Tune Logistic Regression using 5-fold CV on training data only (ROC-AUC).
    """
    param_grid = {
        "max_iter": [50,100,150,200,250]
    }

    model = LogisticRegression(
        solver="liblinear",
        random_state=random_state
    )

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="roc_auc", 
        cv=KFold(n_splits=5, shuffle=True, random_state=random_state),
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train, y_train)

    print(f"    Logistic Regression best params: {grid.best_params_} | CV ROC-AUC={grid.best_score_:.4f}")
    return grid.best_params_, float(grid.best_score_)

def tune_xgboost(X_train, y_train, missing_rate, imputer, random_state=57):
    """
    Tune XGBoost using 5-fold CV on training data only (ROC-AUC).
    """
    param_grid = {
        "max_depth": [3,4,5],
        "subsample": [0.5,0.6,0.7,0.8,0.9,1.0],
        "n_estimators": [50,100,150,200,250,300,350,400],
        "learning_rate": [0.01,0.05,0.1]
    }

    model = XGBClassifier(
        random_state=random_state,
        eval_metric="logloss"
    )

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="roc_auc", 
        cv=KFold(n_splits=5, shuffle=True, random_state=random_state),
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train, y_train)

    print(f"    XGBoost best params: {grid.best_params_} | CV ROC-AUC={grid.best_score_:.4f}")
    return grid.best_params_, float(grid.best_score_)


def tune_mlp(X_train, y_train, missing_rate, imputer, random_state=57):
    """
    Tune MLP (Neural Network) using 5-fold CV on training data only (ROC-AUC).
    """
    param_grid = {
        "learning_rate_init": [0.001,0.01,0.1],
        "hidden_layer_sizes": [(20,), (40,), (60,), (80,), (100,),
                               (20, 20), (40, 40), (60, 60), (80, 80), (100, 100),
                               (20, 20, 20), (40, 40, 40), (60, 60, 60), (80, 80, 80), (100, 100, 100)],
        "alpha": [0.0001, 0.001, 0.01, 0.1]
    }

    model = MLPClassifier(
        random_state=random_state,
        max_iter=500,
        activation="relu",
        solver="adam"
    )

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="roc_auc", 
        cv=KFold(n_splits=5, shuffle=True, random_state=random_state),
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train, y_train)

    print(f"    NeuralNetwork best params: {grid.best_params_} | CV ROC-AUC={grid.best_score_:.4f}")
    return grid.best_params_, float(grid.best_score_)


def evaluate_classifiers(X_train, y_train, X_test, y_test, params, random_state=57):
    """
    Evaluate tuned XGBoost, Neural Network and Logistic Regression classifiers on imputed test datasets
    using Accuracy, F1 and ROC-AUC metrics.
    """
    rows = []

    classifiers = {
        "XGBoost": XGBClassifier(
            random_state=random_state,
            eval_metric="logloss",
            **params["xgb_params"]
        ),
        "NeuralNetwork": MLPClassifier(
            random_state=random_state,
            max_iter=500,
            activation="relu",
            solver="adam",
            **params["mlp_params"]
        ),
        "LogisticRegression": LogisticRegression(
            solver="liblinear",
            random_state=random_state,
            **params["log_params"]
        )
    }

    for clf_name, clf in classifiers.items():
        # Fit
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        # predict probabilities (for ROC-AUC)
        if hasattr(clf, "predict_proba"):
            y_prob = clf.predict_proba(X_test)[:, 1]
        else:
            y_prob = None

        # metrics
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan

        rows.append({
            "classifier": clf_name,
            "accuracy": float(acc),
            "f1": float(f1),
            "roc_auc": float(roc)
        })

    return rows

def run_all(com_dir="data/complete", imp_dir="data/imputed", out_dir="results",
            tuning_csv="classifier_tuning.csv",
            results_csv="classification_results.csv",
            random_state=57):
    
    os.makedirs(out_dir, exist_ok=True)

    tuning_path = os.path.join(out_dir, tuning_csv)
    results_path = os.path.join(out_dir, results_csv)

    tuning_rows = []
    results_rows = []

    for iteration in range(1, 101):
        print(f"\nIteration {iteration}")

        # load complete
        complete_dir = os.path.join(com_dir, f"iteration_{iteration}")
        train_complete = pd.read_csv(os.path.join(complete_dir, "train_complete.csv"))
        test_complete = pd.read_csv(os.path.join(complete_dir, "test_complete.csv"))

        y_train = train_complete["target"].values
        y_test = test_complete["target"].values

        for miss in ["10%", "25%"]:
            print(f"  Missing rate: {miss}")

            for method in ["simple", "knn", "missforest"]:
                print(f"   Imputer: {method}")

                iter_imp_dir = os.path.join(imp_dir, f"iteration_{iteration}")
                train_path = os.path.join(iter_imp_dir, f"train_{miss}_{method}.csv")
                test_path = os.path.join(iter_imp_dir, f"test_{miss}_{method}.csv")

                X_train = pd.read_csv(train_path)
                X_test = pd.read_csv(test_path)

                # tune
                print("    Tuning (ROC-AUC CV) on training set")
                xgb_params, xgb_cv = tune_xgboost(X_train, y_train, miss, method, random_state=random_state)
                mlp_params, mlp_cv = tune_mlp(X_train, y_train, miss, method, random_state=random_state)
                log_params, log_cv = tune_log(X_train, y_train, miss, method, random_state=random_state)

                tuned_params = {"xgb_params": xgb_params, "mlp_params": mlp_params, "log_params": log_params}

                tuning_rows.append({
                    "iteration": iteration,
                    "missing_rate": miss,
                    "imputer": method,
                    "classifier": "XGBoost",
                    "params": json.dumps(xgb_params, sort_keys=True),
                    "cv_roc_auc_best_score": xgb_cv
                })
                tuning_rows.append({
                    "iteration": iteration,
                    "missing_rate": miss,
                    "imputer": method,
                    "classifier": "NeuralNetwork",
                    "params": json.dumps(mlp_params, sort_keys=True),
                    "cv_roc_auc_best_score": mlp_cv
                })
                tuning_rows.append({
                    "iteration": iteration,
                    "missing_rate": miss,
                    "imputer": method,
                    "classifier": "LogisticRegression",
                    "params": json.dumps(log_params, sort_keys=True),
                    "cv_roc_auc_best_score": log_cv
                })

                # evaluate
                print("    Evaluating on test set")
                eval_rows = evaluate_classifiers(
                    X_train=X_train, y_train=y_train,
                    X_test=X_test, y_test=y_test,
                    params=tuned_params,
                    random_state=random_state
                )

                for r in eval_rows:
                    print(f"     {r['classifier']}: ACC={r['accuracy']:.4f} | F1={r['f1']:.4f} | ROC-AUC={r['roc_auc']:.4f}")
                    results_rows.append({
                        "iteration": iteration,
                        "missing_rate": miss,
                        "imputer": method,
                        **r
                    })

        # save afer each iteration
        pd.DataFrame(tuning_rows).to_csv(tuning_path, index=False)
        pd.DataFrame(results_rows).to_csv(results_path, index=False)

    print("\nFinished.")
    print(f"Saved tuning results to {tuning_path}")
    print(f"Saved evaluation results to {results_path}")

    return pd.DataFrame(tuning_rows), pd.DataFrame(results_rows)

if __name__ == "__main__":
    run_all()


Iteration 86
  Missing rate: 10%
   Imputer: simple
    Tuning (ROC-AUC CV) on training set


Exception ignored on calling ctypes callback function: <bound method DataIter._next_wrapper of <xgboost.data.SingleBatchInternalIter object at 0x7f15cce9b150>>
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/xgboost/core.py", line 662, in _next_wrapper
    return self._handle_exception(lambda: int(self.next(input_data)), 0)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/xgboost/core.py", line 575, in _handle_exception
    return fn()
           ^^^^
  File "/opt/conda/lib/python3.11/site-packages/xgboost/core.py", line 662, in <lambda>
    return self._handle_exception(lambda: int(self.next(input_data)), 0)
                                              ^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/xgboost/data.py", line 1642, in next
    input_data(**self.kwargs)
  File "/opt/conda/lib/python3.11/site-packages/xgboost/core.py", line 751, in inner_f
    re

KeyboardInterrupt: 

In [2]:
import os
import json
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, KFold

def tune_log(X_train, y_train, random_state=57):
    """
    Tune Logistic Regression using 5-fold CV on training data only (ROC-AUC).
    """
    param_grid = {
        "max_iter": [50,100,150,200,250]
    }

    model = LogisticRegression(
        solver="liblinear", 
        random_state=random_state
    )

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=KFold(n_splits=5, shuffle=True, random_state=random_state),
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train, y_train)

    print(f"    LogisticRegression best params: {grid.best_params_} | CV ROC-AUC={grid.best_score_:.4f}")
    return grid.best_params_, float(grid.best_score_)


def tune_xgboost(X_train, y_train, random_state=57):
    """
    Tune XGBoost using 5-fold CV on training data only (ROC-AUC).
    """
    param_grid = {
        "max_depth": [3,4,5],
        "subsample": [0.5,0.6,0.7,0.8,0.9,1.0],
        "n_estimators": [50,100,150,200,250,300,350,400],
        "learning_rate": [0.01,0.05,0.1]
    }

    model = XGBClassifier(random_state=random_state, eval_metric="logloss")

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=KFold(n_splits=5, shuffle=True, random_state=random_state),
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train, y_train)

    print(f"    XGBoost best params: {grid.best_params_} | CV ROC-AUC={grid.best_score_:.4f}")
    return grid.best_params_, float(grid.best_score_)


def tune_mlp(X_train, y_train, random_state=57):
    """
    Tune MLP (Neural Network) using 5-fold CV on training data only (ROC-AUC).
    """
    param_grid = {
        "learning_rate_init": [0.001, 0.01, 0.1],
        "hidden_layer_sizes": [(20,), (40,), (60,), (80,), (100,),
                               (20, 20), (40, 40), (60, 60), (80, 80), (100, 100),
                               (20, 20, 20), (40, 40, 40), (60, 60, 60), (80, 80, 80), (100, 100, 100)],
        "alpha": [0.0001, 0.001, 0.01, 0.1]
    }

    model = MLPClassifier(
        random_state=random_state,
        max_iter=500,
        activation="relu",
        solver="adam"
    )

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=KFold(n_splits=5, shuffle=True, random_state=random_state),
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train, y_train)

    print(f"    Neural Network best params: {grid.best_params_} | CV ROC-AUC={grid.best_score_:.4f}")
    return grid.best_params_, float(grid.best_score_)


def evaluate_classifiers_complete(X_train, y_train, X_test, y_test, params, random_state=57):
    """
    Train on complete train, test on complete test (baseline).
    """
    rows = []

    classifiers = {
        "XGBoost": XGBClassifier(
            random_state=random_state,
            eval_metric="logloss",
            **params["xgb_params"]
        ),
        "NeuralNetwork": MLPClassifier(
            random_state=random_state,
            max_iter=500,
            activation="relu",
            solver="adam",
            **params["mlp_params"]
        ),
        "LogisticRegression": LogisticRegression(
            solver="liblinear",
            random_state=random_state,
            **params["log_params"]
        )
    }

    for clf_name, clf in classifiers.items():
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        if hasattr(clf, "predict_proba"):
            y_prob = clf.predict_proba(X_test)[:, 1]
        else:
            y_prob = None

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan

        rows.append({
            "classifier": clf_name,
            "accuracy": float(acc),
            "f1": float(f1),
            "roc_auc": float(roc)
        })

    return rows


def run_all(com_dir="data/complete", out_dir="results",
            tuning_csv="classifier_tuning_complete.csv",
            results_csv="classification_results_complete.csv",
            random_state=57):
    
    os.makedirs(out_dir, exist_ok=True)

    tuning_path = os.path.join(out_dir, tuning_csv)
    results_path = os.path.join(out_dir, results_csv)

    tuning_rows = []
    results_rows = []

    for iteration in range(1, 101):
        print(f"\nIteration {iteration}")

        complete_dir = os.path.join(com_dir, f"iteration_{iteration}")
        train_complete = pd.read_csv(os.path.join(complete_dir, "train_complete.csv"))
        test_complete = pd.read_csv(os.path.join(complete_dir, "test_complete.csv"))

        y_train = train_complete["target"].values
        y_test = test_complete["target"].values

        X_train = train_complete.drop(columns="target")
        X_test = test_complete.drop(columns="target")

        # Tune (train only)
        print("  Tuning (ROC-AUC CV) on complete training set")
        xgb_params, xgb_cv = tune_xgboost(X_train, y_train, random_state=random_state)
        mlp_params, mlp_cv = tune_mlp(X_train, y_train, random_state=random_state)
        log_params, log_cv = tune_log(X_train, y_train, random_state=random_state)

        tuned_params = {"xgb_params": xgb_params, "mlp_params": mlp_params, "log_params": log_params}

        tuning_rows.append({
            "iteration": iteration,
            "classifier": "XGBoost",
            "params": json.dumps(xgb_params, sort_keys=True),
            "cv_roc_auc_best_score": float(xgb_cv)
        })
        tuning_rows.append({
            "iteration": iteration,
            "classifier": "NeuralNetwork",
            "params": json.dumps(mlp_params, sort_keys=True),
            "cv_roc_auc_best_score": float(mlp_cv)
        })
        tuning_rows.append({
            "iteration": iteration,
            "classifier": "LogisticRegression",
            "params": json.dumps(log_params, sort_keys=True),
            "cv_roc_auc_best_score": float(log_cv)
        })

        # Evaluate (test)
        print("  Evaluating on complete test set")
        eval_rows = evaluate_classifiers_complete(
            X_train=X_train, y_train=y_train,
            X_test=X_test, y_test=y_test,
            params=tuned_params,
            random_state=random_state
        )

        for r in eval_rows:
            print(f"   {r['classifier']}: ACC={r['accuracy']:.4f} | F1={r['f1']:.4f} | ROC-AUC={r['roc_auc']:.4f}")
            results_rows.append({
                "iteration": iteration,
                "missing_rate": "complete",
                "imputer": "none",
                **r
            })

        # save after each iteration 
        pd.DataFrame(tuning_rows).to_csv(tuning_path, index=False)
        pd.DataFrame(results_rows).to_csv(results_path, index=False)

    print("\nFinished.")
    print(f"Saved tuning results to {tuning_path}")
    print(f"Saved evaluation results to {results_path}")

    return pd.DataFrame(tuning_rows), pd.DataFrame(results_rows)

if __name__ == "__main__":
    run_all()


Iteration 89
  Tuning (ROC-AUC CV) on complete training set
    XGBoost best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 350, 'subsample': 0.6} | CV ROC-AUC=0.9364


/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptro

    Neural Network best params: {'alpha': 0.1, 'hidden_layer_sizes': (40,), 'learning_rate_init': 0.001} | CV ROC-AUC=0.9813
    LogisticRegression best params: {'max_iter': 50} | CV ROC-AUC=0.9819
  Evaluating on complete test set
   XGBoost: ACC=0.8567 | F1=0.8542 | ROC-AUC=0.9549
   NeuralNetwork: ACC=0.9100 | F1=0.9091 | ROC-AUC=0.9790
   LogisticRegression: ACC=0.9200 | F1=0.9195 | ROC-AUC=0.9830

Iteration 90
  Tuning (ROC-AUC CV) on complete training set
    XGBoost best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 400, 'subsample': 0.7} | CV ROC-AUC=0.9243


/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptro

    Neural Network best params: {'alpha': 0.1, 'hidden_layer_sizes': (20,), 'learning_rate_init': 0.001} | CV ROC-AUC=0.9628
    LogisticRegression best params: {'max_iter': 50} | CV ROC-AUC=0.9706
  Evaluating on complete test set


...

    Neural Network best params: {'alpha': 0.1, 'hidden_layer_sizes': (40,), 'learning_rate_init': 0.001} | CV ROC-AUC=0.9624
    LogisticRegression best params: {'max_iter': 50} | CV ROC-AUC=0.9748
  Evaluating on complete test set
   XGBoost: ACC=0.8400 | F1=0.8356 | ROC-AUC=0.9336
   NeuralNetwork: ACC=0.9000 | F1=0.8986 | ROC-AUC=0.9596
   LogisticRegression: ACC=0.8867 | F1=0.8859 | ROC-AUC=0.9704

Iteration 99
  Tuning (ROC-AUC CV) on complete training set
    XGBoost best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 350, 'subsample': 0.5} | CV ROC-AUC=0.9585


/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptro

    Neural Network best params: {'alpha': 0.1, 'hidden_layer_sizes': (20,), 'learning_rate_init': 0.001} | CV ROC-AUC=0.9790
    LogisticRegression best params: {'max_iter': 50} | CV ROC-AUC=0.9857
  Evaluating on complete test set
   XGBoost: ACC=0.8533 | F1=0.8533 | ROC-AUC=0.9485
   NeuralNetwork: ACC=0.9167 | F1=0.9147 | ROC-AUC=0.9805
   LogisticRegression: ACC=0.9033 | F1=0.8990 | ROC-AUC=0.9800

Iteration 100
  Tuning (ROC-AUC CV) on complete training set
    XGBoost best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 400, 'subsample': 0.5} | CV ROC-AUC=0.9553


/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptro

    Neural Network best params: {'alpha': 0.1, 'hidden_layer_sizes': (100,), 'learning_rate_init': 0.001} | CV ROC-AUC=0.9805
    LogisticRegression best params: {'max_iter': 50} | CV ROC-AUC=0.9849
  Evaluating on complete test set
   XGBoost: ACC=0.8700 | F1=0.8713 | ROC-AUC=0.9464
   NeuralNetwork: ACC=0.9267 | F1=0.9262 | ROC-AUC=0.9833
   LogisticRegression: ACC=0.9533 | F1=0.9530 | ROC-AUC=0.9874

Finished.
Saved tuning results to results/classifier_tuning_complete.csv
Saved evaluation results to results/classification_results_complete.csv
